## Model components

In [1]:
from pyomo.environ import *
import numpy as np
import math
import random

## Create 30 senarios

In [2]:
# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

## Model components

# Generate 30 scenarios with random data
scenarios = list(range(1, 31))  # 1 to 30 scenarios

# Generate random probabilities that sum to 1
prob_values = np.random.dirichlet(np.ones(30))
prob = {i+1: prob_values[i] for i in range(30)}

# Generate random demand data for each scenario
d = {}
for s in scenarios:
    d[s] = {
        1: random.randint(100, 500),  # demand for product 1: between 100-500
        2: random.randint(200, 400)   # demand for product 2: between 200-400
    }

# Generate random profit coefficients for each scenario
q = {}
for s in scenarios:
    q[s] = {
        1: random.randint(-30, -20),  # profit coefficient for product 1: between -30 to -20
        2: random.randint(-35, -25)   # profit coefficient for product 2: between -35 to -25
    }

print("Generated Data:")
print(f"Number of scenarios: {len(scenarios)}")
print(f"First 5 scenarios demands: {[(s, d[s]) for s in scenarios[:5]]}")
print(f"First 5 scenarios profits: {[(s, q[s]) for s in scenarios[:5]]}")
print(f"First 5 probabilities: {[(s, prob[s]) for s in scenarios[:5]]}")

Generated Data:
Number of scenarios: 30
First 5 scenarios demands: [(1, {1: 427, 2: 228}), (2, {1: 112, 2: 389}), (3, {1: 240, 2: 262}), (4, {1: 214, 2: 235}), (5, {1: 477, 2: 226})]
First 5 scenarios profits: [(1, {1: -29, 2: -27}), (2, {1: -26, 2: -25}), (3, {1: -21, 2: -30}), (4, {1: -21, 2: -32}), (5, {1: -29, 2: -35})]
First 5 probabilities: [(1, np.float64(0.01941209198872349)), (2, np.float64(0.12451891650480727)), (3, np.float64(0.054469479333487704)), (4, np.float64(0.03776545904749857)), (5, np.float64(0.0070168282466271))]


# H&T

In [3]:
## Create T and h matrices for all scenarios
T = {}
h = {}

for s in scenarios:
    # Create a temporary subproblem to extract coefficients
    sub = ConcreteModel()
    sub.y = Var(RangeSet(2), within=NonNegativeReals)
    sub.x = Var(RangeSet(2), within=NonNegativeReals)  # Temporary x variables

    sub.c1 = Constraint(expr=6*sub.y[1] + 10*sub.y[2] <= 60 * sub.x[1])
    sub.c2 = Constraint(expr=8*sub.y[1] + 5*sub.y[2] <= 80 * sub.x[2])
    sub.c3 = Constraint(expr=sub.y[1] <= d[s][1])
    sub.c4 = Constraint(expr=sub.y[2] <= d[s][2])

    # Extract T matrix (coefficients of x variables)
    T_tmp = []
    T_tmp.append([-60, 0])    # coefficient of x in constraint 1
    T_tmp.append([0, -80])    # coefficient of x in constraint 2
    T_tmp.append([0, 0])      # no x in third constraint
    T_tmp.append([0, 0])      # no x in fourth constraint

    T[s] = T_tmp

    # Extract h vector (RHS values)
    h_tmp = []
    h_tmp.append(0.0)         # RHS of constraint 1 (becomes 0 after moving x terms)
    h_tmp.append(0.0)         # RHS of constraint 2 (becomes 0 after moving x terms)
    h_tmp.append(float(d[s][1]))  # RHS of constraint 3
    h_tmp.append(float(d[s][2]))  # RHS of constraint 4

    h[s] = h_tmp

print(f"\nT matrix for scenario 1: {T[1]}")
print(f"h vector for scenario 1: {h[1]}")


T matrix for scenario 1: [[-60, 0], [0, -80], [0, 0], [0, 0]]
h vector for scenario 1: [0.0, 0.0, 427.0, 228.0]


## Testing Algorithm For 30 scenarios

In [4]:

## L-Shaped Algorithm Implementation

# Initialize counters
r, s_counter, v = (0, 0, 0)
theta_values = []
w_values = []
max_iterations = 20  # Maximum number of iterations to prevent infinite loop
tolerance = 1e-6     # Convergence tolerance

# Step 1: Define Master problem
model = ConcreteModel()
model.I = RangeSet(2)
model.x = Var(model.I, domain=NonNegativeReals)
model.theta = Var(scenarios, domain=Reals)

# Objective: original cost + expected future cost for all scenarios
model.obj = Objective(expr=100*model.x[1] + 150*model.x[2] + sum(model.theta[s] for s in scenarios))

# Master problem constraints
model.con1 = Constraint(expr=model.x[1] + model.x[2] <= 120)
model.con2 = Constraint(expr=model.x[1] >= 40)
model.con3 = Constraint(expr=model.x[2] >= 20)
model.optimal_cut = ConstraintList()



not_stop_condition = True
while not_stop_condition and v < max_iterations:
    not_stop_conditionF = True
    while not_stop_conditionF:

        v += 1
        if v == 1:
            # Initialize theta values for all scenarios
            model.theta_constraints = ConstraintList()
            for s in scenarios:
                model.theta_constraints.add(model.theta[s] == -10000000)
                theta_values.append(-10000000)
        else:
            # Deactivate previous theta constraints
            for constraint in model.theta_constraints.values():
                constraint.deactivate()

            print(f"\n[Theta values from previous iteration updated]")

        solver = SolverFactory('cplex')
        result = solver.solve(model, tee=False)

        print("\n" + "="*50)
        print(f"🔁 Iteration {v}")
        print("-"*50)
        print(f"📌 Master Problem Objective Value : {value(model.obj):,.2f}")
        print(f"📌 Decision Variables:")
        print(f"    x1 = {value(model.x[1]):.2f}")
        print(f"    x2 = {value(model.x[2]):.2f}")
        print("="*50)

        # Step 2: Feasibility check for all scenarios
        di = []
        D = []
        simplex_multiplierF = {}

        for s in scenarios:
            feas = ConcreteModel()
            feas.y = Var(model.I, within=NonNegativeReals)
            feas.v_plus = Var(RangeSet(4), domain=NonNegativeReals)
            feas.v_minus = Var(RangeSet(4), domain=NonNegativeReals)

            feas.obj = Objective(expr=sum(feas.v_plus[i] + feas.v_minus[i] for i in RangeSet(4)), sense=minimize)

            feas.c1 = Constraint(expr=6*feas.y[1] + 10*feas.y[2] + feas.v_plus[1] - feas.v_minus[1] <= 60 * value(model.x[1]))
            feas.c2 = Constraint(expr=8*feas.y[1] + 5*feas.y[2] + feas.v_plus[2] - feas.v_minus[2] <= 80 * value(model.x[2]))
            feas.c3 = Constraint(expr=feas.y[1] + feas.v_plus[3] - feas.v_minus[3] <= d[s][1])
            feas.c4 = Constraint(expr=feas.y[2] + feas.v_plus[4] - feas.v_minus[4] <= d[s][2])

            # For GLPK, dual values might not be available, so we'll handle this differently
            try:
                feas.dual = Suffix(direction=Suffix.IMPORT)
                solver = SolverFactory('cplex')
                result = solver.solve(feas, tee=False)

                dual_var_F = []
                if hasattr(feas, 'dual'):
                    for c in feas.component_objects(Constraint, active=True):
                        cobject = getattr(feas, str(c))
                        for index in cobject:
                            try:
                                dual_var_F.append(feas.dual[cobject[index]])
                            except KeyError:
                                dual_var_F.append(0.0)  # Default to 0 if dual not available
                else:
                    dual_var_F = [0.0, 0.0, 0.0, 0.0]  # Default dual values

                simplex_multiplierF[s] = dual_var_F
            except:
                # If dual values are not available, use default values
                simplex_multiplierF[s] = [0.0, 0.0, 0.0, 0.0]

            di.append(prob[s] * np.array([simplex_multiplierF[s]]) @ np.array(h[s]))
            D.append(prob[s] * np.array([simplex_multiplierF[s]]) @ np.array(T[s]))

        print(f"\n🧮 Feasibility check completed for all {len(scenarios)} scenarios")
        print('---'*10)

        # Check feasibility
        zero = 0
        feasibility_violated = any(value(feas.obj) > zero for _ in scenarios)

        if feasibility_violated:
            r += 1
            # Add feasibility cuts (if needed)
            model.optimal_cut.add(list(D @ np.array([model.x[1], model.x[2]]))[0] + model.theta[s] >= di[0])  
            print("\n✅ Feasibility cuts would be added here if violated")
            print("="*50 + "\n")
        else:
            print("\n🛑 Feasibility is OK for all scenarios!")
            print("="*50 + "\n")
            not_stop_conditionF = False

    #  Solve subproblems for all scenarios
    e = []
    E = []
    simplex_multiplier = {}

    for s in scenarios:
        sub = ConcreteModel()
        sub.y = Var(model.I, within=NonNegativeReals)
        sub.obj = Objective(expr=q[s][1]*sub.y[1] + q[s][2]*sub.y[2], sense=minimize)

        sub.c1 = Constraint(expr=6*sub.y[1] + 10*sub.y[2] <= 60 * value(model.x[1]))
        sub.c2 = Constraint(expr=8*sub.y[1] + 5*sub.y[2] <= 80 * value(model.x[2]))
        sub.c3 = Constraint(expr=sub.y[1] <= d[s][1])
        sub.c4 = Constraint(expr=sub.y[2] <= d[s][2])

        try:
            sub.dual = Suffix(direction=Suffix.IMPORT)
            solver = SolverFactory('cplex')
            result = solver.solve(sub, tee=False)

            dual_var_2 = []
            if hasattr(sub, 'dual'):
                for c in sub.component_objects(Constraint, active=True):
                    cobject = getattr(sub, str(c))
                    for index in cobject:
                        try:
                            dual_var_2.append(sub.dual[cobject[index]])
                        except KeyError:
                            dual_var_2.append(0.0)  # Default to 0 if dual not available
            else:
                dual_var_2 = [0.0, 0.0, 0.0, 0.0]  # Default dual values

            simplex_multiplier[s] = dual_var_2
        except:
            # If dual values are not available, use default values
            simplex_multiplier[s] = [0.0, 0.0, 0.0, 0.0]

        e.append(prob[s] * np.array([simplex_multiplier[s]]) @ np.array(h[s]))
        E.append(prob[s] * np.array([simplex_multiplier[s]]) @ np.array(T[s]))

    print(f"\n🧮 Subproblems solved for all {len(scenarios)} scenarios")
    print('---'*10)

    # Step 3: Check optimality and add cuts
    cuts_added = False
    max_violation = 0.0

    for s in scenarios:
        w = np.array(h[s]) - np.array(T[s]) @ np.array([value(model.x[1]), value(model.x[2])])
        dot_product = prob[s] * np.dot(simplex_multiplier[s], w)

        violation = dot_product - value(model.theta[s])
        max_violation = max(max_violation, violation)

        if violation > tolerance:  # Use tolerance instead of direct comparison
            cuts_added = True

    print(f"\n📊 Maximum violation: {max_violation:.6f}")
    print(f"📊 Tolerance: {tolerance}")

    if cuts_added and max_violation > tolerance:
        # Add optimality cuts for all scenarios automatically
        for s in scenarios:
            w = np.array(h[s]) - np.array(T[s]) @ np.array([value(model.x[1]), value(model.x[2])])
            dot_product = prob[s] * np.dot(simplex_multiplier[s], w)

            if dot_product - value(model.theta[s]) > tolerance:
                model.optimal_cut.add(
                    sum(E[s-1][0][j] * model.x[j+1] for j in range(2)) + model.theta[s] >= float(e[s-1])
                )

        print(f"\n✅ Added optimality cuts to master problem.")
        print(f"📊 Total cuts in model: {len(model.optimal_cut)}")
        print("="*50 + "\n")
    else:
        # Final solution
        solver = SolverFactory('cplex')
        result = solver.solve(model, tee=False)

        print("\n" + "="*50)
        print(f"🔁 Final Iteration {v}")
        print("-"*50)
        print(f"📌 Master Problem Objective Value : {value(model.obj):,.2f}")
        print(f"📌 Decision Variables:")
        print(f"    x1 = {value(model.x[1]):.2f}")
        print(f"    x2 = {value(model.x[2]):.2f}")

        # Print theta values for first few scenarios as sample
        print(f"📌 Sample Theta Values (first 5 scenarios):")
        for s in scenarios[:5]:
            print(f"    θ[{s}] = {value(model.theta[s]):.2f}")

        print("="*50)
        print("\n🛑 Termination Criteria Met!")
        print(f"✅ Final Master Problem Objective: {value(model.obj):,.2f}")
        print(f"✅ Final Decision Variables: x1 = {value(model.x[1]):.2f}, x2 = {value(model.x[2]):.2f}")
        print("="*50 + "\n")
        print(f"✅ Final counter Variables: r = {r}, s = {s_counter}, v = {v}")
        print(f"✅ Total scenarios processed: {len(scenarios)}")
        print(f"✅ Total optimality cuts added: {len(model.optimal_cut)}")
        not_stop_condition = False

# Check if we stopped due to max iterations
if v >= max_iterations:
    print(f"\n⚠️  Algorithm stopped after {max_iterations} iterations (maximum limit reached)")
    print(f"⚠️  Consider increasing max_iterations if more precision is needed")


🔁 Iteration 1
--------------------------------------------------
📌 Master Problem Objective Value : -299,993,000.00
📌 Decision Variables:
    x1 = 40.00
    x2 = 20.00

🧮 Feasibility check completed for all 30 scenarios
------------------------------

🛑 Feasibility is OK for all scenarios!


🧮 Subproblems solved for all 30 scenarios
------------------------------

📊 Maximum violation: 9999992.827709
📊 Tolerance: 1e-06

✅ Added optimality cuts to master problem.
📊 Total cuts in model: 30


[Theta values from previous iteration updated]


C:\Users\mmm\AppData\Local\Temp\ipykernel_13868\2289672073.py:188: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  sum(E[s-1][0][j] * model.x[j+1] for j in range(2)) + model.theta[s] >= float(e[s-1])



🔁 Iteration 2
--------------------------------------------------
📌 Master Problem Objective Value : -2,343.71
📌 Decision Variables:
    x1 = 100.00
    x2 = 20.00

🧮 Feasibility check completed for all 30 scenarios
------------------------------

🛑 Feasibility is OK for all scenarios!


🧮 Subproblems solved for all 30 scenarios
------------------------------

📊 Maximum violation: 1170.859361
📊 Tolerance: 1e-06

✅ Added optimality cuts to master problem.
📊 Total cuts in model: 60


[Theta values from previous iteration updated]

🔁 Iteration 3
--------------------------------------------------
📌 Master Problem Objective Value : -714.55
📌 Decision Variables:
    x1 = 70.78
    x2 = 49.22

🧮 Feasibility check completed for all 30 scenarios
------------------------------

🛑 Feasibility is OK for all scenarios!


🧮 Subproblems solved for all 30 scenarios
------------------------------

📊 Maximum violation: 343.539908
📊 Tolerance: 1e-06

✅ Added optimality cuts to master problem.
📊 Total cut